<a href="https://colab.research.google.com/github/A-J-Jovia/jovia-codeboosters-2026/blob/main/day6/Miniproj_day6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# GROQ: https://console.groq.com/home

!pip install groq --quiet

import os
import json
import re
import time
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported")

Libraries imported


In [3]:
from groq import Groq
from google.colab import userdata

API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=API_KEY)
MODEL = "llama-3.1-8b-instant"
print(f"Groq Client configured with model: {MODEL}")

Groq Client configured with model: llama-3.1-8b-instant


In [4]:
def ask_llm(user_message, system_message="You are a helpful assistant", temperature=0.7, max_tokens=500):
  response = client.chat.completions.create(
    model=MODEL,
    messages=[
      {
        "role": "system",
        "content": system_message
      },
      {
        "role": "user",
        "content": user_message
      }],
    temperature = temperature,
    max_tokens = max_tokens,
  )
  return response.choices[0].message.content

test_response = ask_llm("What is the meaning of the name 'Jovia'?")
print("=== RESPONSE ===\n",test_response)

=== RESPONSE ===
 The name 'Jovia' is derived from the Latin word 'jovialis,' which means 'relating to Jupiter' or 'Jovian.' In Roman mythology, Jupiter was the king of the gods and the god of the sky and thunder. 

As a given name, 'Jovia' likely conveys a sense of grandeur, power, and wisdom, reflecting the qualities associated with the Roman god Jupiter. It may also symbolize a connection to the divine, the celestial, or the infinite.

In terms of cultural and historical context, 'Jovia' might be associated with the Renaissance and the classical revival in art and literature, where the mythology of ancient Rome and Greece was widely studied and emulated.

However, it's worth noting that 'Jovia' is not a commonly used given name in modern times.


In [5]:
messy_invoices = [
    "INV-2024-0091 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase",
    "Invoice from PRIYA ENTIREPRICES dt 07-02-2024 amt: 12500 for Office Cleaning services",
    "#INV-2024-103 | arjun nair consultancy | 5000 | march 15 2024 | python training",
    "SURESH RAO HARDWARE STORE 2500 Keyboard and mouse accessories 2024/01/10",
    "Tax Invoice:Ananya Tech Solutions | Inv-897|Date:28-feb-24 | Amount:INR 95,000|Server hardware"
]

print('MEssy invoices to process:')
for i, inv in enumerate(messy_invoices,1):
  print(f'{i+1}.{inv}')
print(f'\nTotal: {len(messy_invoices)}invoices')

MEssy invoices to process:
2.INV-2024-0091 TECHWORLD SOLUTIONS 15th Jan 2024 Rs.45,000 Laptop purchase
3.Invoice from PRIYA ENTIREPRICES dt 07-02-2024 amt: 12500 for Office Cleaning services
4.#INV-2024-103 | arjun nair consultancy | 5000 | march 15 2024 | python training
5.SURESH RAO HARDWARE STORE 2500 Keyboard and mouse accessories 2024/01/10
6.Tax Invoice:Ananya Tech Solutions | Inv-897|Date:28-feb-24 | Amount:INR 95,000|Server hardware

Total: 5invoices


In [10]:
import json

print("Processing Invoices with LLM...")
print("="*100)

user_invoice_message = "\n".join(messy_invoices)

llm_raw_response = ask_llm(user_invoice_message,
                            system_message='''Clean the data and organize data in same structure
                            (company,amount, invoice_date(YYYY-MM-DD), product)
                            return as JSON.
                            Only output the JSON.
                            No code.''')

print("==== Raw LLM Response ====")
print(llm_raw_response,'\n')

try:
  extracted_records = json.loads(llm_raw_response)
  print("Successfully parsed LLM response to a list of dictionaries.")
except json.JSONDecodeError as e:
  print(f"Error decoding JSON from LLM response: {e}")
  print("LLM Raw Response (unparseable):\n", llm_raw_response)
  extracted_records = []
except Exception as e:
  print(f"An unexpected error occurred during parsing: {e}")
  extracted_records = []


Processing Invoices with LLM...
==== Raw LLM Response ====
[
  {
    "company": "TECHWORLD SOLUTIONS",
    "amount": 45000,
    "invoice_date": "2024-01-15",
    "product": "Laptop purchase"
  },
  {
    "company": "PRIYA ENTIREPRICES",
    "amount": 12500,
    "invoice_date": "2024-02-07",
    "product": "Office Cleaning services"
  },
  {
    "company": "arjun nair consultancy",
    "amount": 5000,
    "invoice_date": "2024-03-15",
    "product": "python training"
  },
  {
    "company": "SURESH RAO HARDWARE STORE",
    "amount": 2500,
    "invoice_date": "2024-01-10",
    "product": "Keyboard and mouse accessories"
  },
  {
    "company": "Ananya Tech Solutions",
    "amount": 95000,
    "invoice_date": "2024-02-28",
    "product": "Server hardware"
  }
] 

Successfully parsed LLM response to a list of dictionaries.


In [11]:
invoices_df = pd.DataFrame(extracted_records)
invoices_df['amount'] = pd.to_numeric(invoices_df['amount'],errors='coerce')
invoices_df['invoice_date'] = pd.to_datetime(invoices_df['invoice_date'],errors='coerce')

print("=== SMART DATA CLEANER OUTPUT ===")
print(f"Rows: {len(invoices_df)} | Columns: {len(invoices_df.columns)}\n")
print(invoices_df.to_string(index=False))

=== SMART DATA CLEANER OUTPUT ===
Rows: 5 | Columns: 4

                  company  amount invoice_date                        product
      TECHWORLD SOLUTIONS   45000   2024-01-15                Laptop purchase
       PRIYA ENTIREPRICES   12500   2024-02-07       Office Cleaning services
   arjun nair consultancy    5000   2024-03-15                python training
SURESH RAO HARDWARE STORE    2500   2024-01-10 Keyboard and mouse accessories
    Ananya Tech Solutions   95000   2024-02-28                Server hardware
